# Whisper Large-v3 — locked TF-IDF/Ridge comparison

# SHL 2026 — TF-IDF + Ridge experiment notebook

A CPU baseline using the **saved Whisper transcripts**, with **six-fold validation**,
a single hyperparameter cell, out-of-fold diagnostics, full-data training, and
`submission.csv`. No audio is retranscribed. The private transcript dataset and
competition data are attached; internet can remain off.

Work through the sections in order for the first run. Then change the hyperparameters
and rerun from the model cell onward. Save each experiment's outputs to compare RMSE.

## 1. Hyperparameters — edit this cell

The initial baseline preserves grammatical mistakes. The default six folds are
shuffled KFold because the CSV has no speaker identifiers. If you later discover
speaker groups, replace the splitter with GroupKFold before comparing experiments.
The observed training labels span 0–5 and include half-point scores. Prediction
clipping below uses that observed range; set `clip_predictions=False` to compare.

In [ ]:
CONFIG = {'experiment_name': 'tfidf_ridge_large_v3', 'seed': 42, 'n_splits': 6, 'ridge_alpha': 10.0, 'ridge_solver': 'lsqr', 'ridge_tol': 1e-05, 'ridge_max_iter': 5000, 'use_word_tfidf': True, 'use_char_tfidf': True, 'word_ngram_range': (1, 2), 'char_ngram_range': (3, 5), 'word_max_features': 40000, 'char_max_features': 60000, 'min_df': 2, 'max_df': 1.0, 'lowercase': False, 'sublinear_tf': True, 'word_weight': 1.0, 'char_weight': 1.0, 'clip_predictions': True, 'prediction_min': 0.0, 'prediction_max': 5.0, 'save_fold_models': False}
LOCAL_LABELS_DIR = 'artifacts/whisper_comparison/reference'
LOCAL_TRANSCRIPTS_PATH = 'artifacts/transcription_large_v3/transcripts_large_v3.parquet'

## 2. Imports and output directory

In [ ]:
from pathlib import Path
import json
import platform
import time
import warnings
import joblib
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, FileLink
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import FeatureUnion, Pipeline

IS_KAGGLE = Path('/kaggle/input').exists()
OUTPUT_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path('artifacts/baseline')
OUTPUT_DIR = OUTPUT_ROOT / CONFIG['experiment_name']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert CONFIG['n_splits'] == 6, 'This experiment is configured for six-fold validation.'
assert CONFIG['use_word_tfidf'] or CONFIG['use_char_tfidf'], 'Enable at least one text representation.'
print('Output directory:', OUTPUT_DIR)
print('scikit-learn:', sklearn.__version__)
display(pd.Series(CONFIG, name='value').to_frame())

## 3. Load transcripts and competition labels

Training and test filenames overlap, so join using the **split first**, then filename.
Test labels of `-1` are placeholders and never enter training or validation.

In [ ]:
if IS_KAGGLE:
    input_root = Path('/kaggle/input')
    label_candidates = [p for p in input_root.rglob('train.csv')
                        if 'shl-hiring-assessment-2026' in str(p)]
    transcript_candidates = [p for p in input_root.rglob('transcripts.parquet')
                             if 'shl-2026-whisper-large-v3-transcript-data' in str(p)]
    assert len(label_candidates) == 1, 'Attach the SHL Hiring Assessment 2026 competition.'
    assert len(transcript_candidates) == 1, 'Attach the private shl-2026-whisper-large-v3-transcript-data dataset.'
    labels_dir = label_candidates[0].parent
    transcripts_path = transcript_candidates[0]
else:
    labels_dir = Path(LOCAL_LABELS_DIR)
    transcripts_path = Path(LOCAL_TRANSCRIPTS_PATH)

train_labels = pd.read_csv(labels_dir / 'train.csv')
test_labels = pd.read_csv(labels_dir / 'test.csv')
sample_submission = pd.read_csv(labels_dir / 'sample_submission.csv')
transcripts = pd.read_parquet(transcripts_path)
assert {'filename', 'label'}.issubset(train_labels.columns)
assert {'audio_id', 'split', 'transcript', 'error'}.issubset(transcripts.columns)
assert not transcripts.duplicated(['split', 'audio_id']).any()
assert transcripts.error.isna().all(), 'Resolve transcription errors before training.'
assert transcripts.transcript.fillna('').str.strip().ne('').all()

train = train_labels.merge(
    transcripts.loc[transcripts['split'].eq('train')],
    left_on='filename', right_on='audio_id', how='left', validate='one_to_one',
)
test = test_labels.merge(
    transcripts.loc[transcripts['split'].eq('test')],
    left_on='filename', right_on='audio_id', how='left', validate='one_to_one',
)
assert len(train) == 769 and len(test) == 216
assert train.transcript.notna().all() and test.transcript.notna().all()
assert train.label.notna().all() and np.isfinite(train.label).all()
assert not train.filename.duplicated().any() and not test.filename.duplicated().any()
print(f'Train: {len(train)} | Test: {len(test)} | Label range: {train.label.min()}–{train.label.max()}')
print('Label values:', sorted(train.label.unique()))
display(train[['filename', 'label', 'transcript']].head())

## 4. Minimal text preprocessing and data inspection

Normalize whitespace only. Preserve grammatical mistakes, function words, contractions,
filler words, and repetitions. The lowercase option is a vectorizer experiment, not
an automatic grammar correction. This TF-IDF baseline uses complete transcripts.

In [ ]:
train['text'] = train.transcript.str.replace(r'\s+', ' ', regex=True).str.strip()
test['text'] = test.transcript.str.replace(r'\s+', ' ', regex=True).str.strip()
assert train.text.ne('').all() and test.text.ne('').all()
train['word_count'] = train.text.str.split().str.len()
test['word_count'] = test.text.str.split().str.len()
print('Exact duplicate normalized training texts:', train.text.duplicated().sum())
if train.text.duplicated().any():
    warnings.warn('Duplicate texts detected: group duplicates before interpreting validation.')
display(train[['label', 'word_count']].describe())
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
train.label.value_counts().sort_index().plot.bar(ax=axes[0], title='Training score counts')
axes[0].set_xlabel('Grammar score')
axes[1].hist(train.word_count, bins=30)
axes[1].set_title('Transcript lengths'); axes[1].set_xlabel('Words')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'data_overview.png', dpi=140); plt.show()

## 5. Model factory — replace or extend here

Word n-grams capture short phrases; character n-grams provide another representation
of spelling and local word structure. Ridge predicts a continuous score.
Each fold gets a **new pipeline**, so vocabulary and IDF are fitted only on that
fold's training rows. There is no global vocabulary fitted before validation.

In [ ]:
def build_model(config):
    branches = []
    weights = {}
    common = dict(min_df=config['min_df'], max_df=config['max_df'],
                  lowercase=config['lowercase'], sublinear_tf=config['sublinear_tf'],
                  dtype=np.float32)
    if config['use_word_tfidf']:
        branches.append(('word', TfidfVectorizer(
            analyzer='word', token_pattern=r'(?u)\b\w+\b',
            ngram_range=config['word_ngram_range'],
            max_features=config['word_max_features'], **common)))
        weights['word'] = config['word_weight']
    if config['use_char_tfidf']:
        branches.append(('char', TfidfVectorizer(
            analyzer='char_wb', ngram_range=config['char_ngram_range'],
            max_features=config['char_max_features'], **common)))
        weights['char'] = config['char_weight']
    return Pipeline([
        ('tfidf', FeatureUnion(branches, transformer_weights=weights)),
        ('ridge', Ridge(alpha=config['ridge_alpha'], solver=config['ridge_solver'],
                        tol=config['ridge_tol'], max_iter=config['ridge_max_iter'])),
    ])


def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))


def postprocess(predictions):
    predictions = np.asarray(predictions, dtype=float)
    if CONFIG['clip_predictions']:
        predictions = np.clip(predictions, CONFIG['prediction_min'], CONFIG['prediction_max'])
    return predictions

X = train['text']
y = train['label'].to_numpy(dtype=float)
print(build_model(CONFIG))

## 6. Six-fold cross-validation

Every training row receives one prediction from a model that did not train on that
row. Compare pooled out-of-fold RMSE, not training RMSE. The reference predictor is
the training-fold mean score. Folds stay fixed across hyperparameter experiments.

In [ ]:
splitter = KFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=CONFIG['seed'])
oof_raw = np.full(len(train), np.nan)
oof_mean = np.full(len(train), np.nan)
fold_ids = np.full(len(train), -1, dtype=int)
fold_results = []
started = time.perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(splitter.split(X), start=1):
    model = build_model(CONFIG)
    fold_started = time.perf_counter()
    model.fit(X.iloc[fit_idx], y[fit_idx])
    raw = model.predict(X.iloc[valid_idx])
    oof_raw[valid_idx] = raw
    oof_mean[valid_idx] = y[fit_idx].mean()
    fold_ids[valid_idx] = fold
    result = {'fold': fold, 'train_rows': len(fit_idx), 'valid_rows': len(valid_idx),
              'rmse': rmse(y[valid_idx], postprocess(raw)),
              'raw_rmse': rmse(y[valid_idx], raw),
              'mean_baseline_rmse': rmse(y[valid_idx], oof_mean[valid_idx]),
              'seconds': time.perf_counter() - fold_started}
    fold_results.append(result)
    if CONFIG['save_fold_models']:
        joblib.dump(model, OUTPUT_DIR / f'fold_{fold}_model.joblib', compress=3)
    print(f"Fold {fold}/6 | RMSE {result['rmse']:.5f} | mean baseline {result['mean_baseline_rmse']:.5f} | {result['seconds']:.1f}s", flush=True)

assert np.isfinite(oof_raw).all() and (fold_ids > 0).all()
cv_metrics = pd.DataFrame(fold_results)
oof_predictions = postprocess(oof_raw)
summary = {'oof_rmse': rmse(y, oof_predictions), 'raw_oof_rmse': rmse(y, oof_raw),
           'mean_baseline_oof_rmse': rmse(y, oof_mean),
           'fold_rmse_mean': float(cv_metrics.rmse.mean()),
           'fold_rmse_std': float(cv_metrics.rmse.std(ddof=1)),
           'cv_seconds': time.perf_counter() - started}
cv_metrics.to_csv(OUTPUT_DIR / 'cv_metrics.csv', index=False)
display(cv_metrics)
print('Pooled six-fold results:', json.dumps(summary, indent=2))

## 7. Out-of-fold diagnostics

Inspect the largest errors by listening to the original recordings when needed.
Do not change labels automatically. Repeated tuning against these same folds can
also overfit validation; keep a record of each experiment.

In [ ]:
oof = train[['filename', 'label', 'transcript', 'word_count']].copy()
oof['fold'] = fold_ids
oof['prediction_raw'] = oof_raw
oof['prediction'] = oof_predictions
oof['residual'] = oof['label'] - oof['prediction']
oof['absolute_error'] = oof['residual'].abs()
oof.to_csv(OUTPUT_DIR / 'oof_predictions.csv', index=False)
display(oof.groupby('label').agg(rows=('filename', 'size'),
                                mean_prediction=('prediction', 'mean'),
                                mean_absolute_error=('absolute_error', 'mean')))
display(oof.nlargest(10, 'absolute_error')[['filename', 'label', 'prediction', 'absolute_error', 'transcript']])
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].scatter(y, oof_predictions, alpha=0.35)
axes[0].plot([0, 5], [0, 5], 'k--')
axes[0].set(xlabel='True score', ylabel='OOF prediction', title='Out-of-fold predictions')
axes[1].hist(oof.residual, bins=30)
axes[1].set(xlabel='True minus predicted', title='Residual distribution')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'oof_diagnostics.png', dpi=140); plt.show()

## 8. Train on all 769 labelled recordings and predict the test set

Validation models estimate performance. This final model is refitted on the complete
training set, then predicts all 216 test transcripts. Test placeholders never enter
model fitting. The saved pipeline contains the fitted vectorizers and Ridge model;
apply the same whitespace normalization when reusing it.

In [ ]:
final_model = build_model(CONFIG)
final_model.fit(X, y)
test_raw = final_model.predict(test['text'])
test_predictions = postprocess(test_raw)
assert len(test_predictions) == len(test) and np.isfinite(test_predictions).all()
joblib.dump(final_model, OUTPUT_DIR / 'full_train_model.joblib', compress=3)
prediction_table = pd.DataFrame({'filename': test.filename.to_numpy(),
                                 'prediction_raw': test_raw, 'label': test_predictions})
prediction_table.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
display(prediction_table.head(10))
print('Test prediction range:', test_predictions.min(), test_predictions.max())

## 9. Build and validate submission.csv

The currently downloaded template contains 204 rows whose IDs differ from the
216 rows in `test.csv`. This cell checks rather than assumes that the template
matches. When IDs match, it preserves template order. Otherwise it uses the actual
test IDs in test.csv order and reports the discrepancy. Never fill missing template
IDs with arbitrary scores. Kaggle's submission validation decides whether those
actual test IDs match its evaluator.

In [ ]:
assert list(sample_submission.columns) == ['filename', 'label'], 'Competition submission schema changed.'
template_matches = (len(sample_submission) == len(test)
                    and not sample_submission.filename.duplicated().any()
                    and set(sample_submission.filename) == set(test.filename))
if template_matches:
    submission = sample_submission[['filename']].merge(
        prediction_table[['filename', 'label']], on='filename', how='left', validate='one_to_one')
else:
    warnings.warn(f'Sample submission has {len(sample_submission)} rows; test.csv has {len(test)}. '
                  'Template IDs differ. Using the actual test.csv identifiers.')
    submission = prediction_table[['filename', 'label']].copy()
assert len(submission) == len(test)
assert not submission.filename.duplicated().any()
assert set(submission.filename) == set(test.filename)
assert np.isfinite(submission.label).all()
submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
# Also expose a standard top-level path for the Kaggle submission picker.
submission.to_csv(OUTPUT_ROOT / 'submission.csv', index=False)
manifest = {'config': CONFIG, 'metrics': summary, 'train_rows': len(train),
            'test_rows': len(test), 'template_matches_test': bool(template_matches),
            'transcripts_dataset': 'prabalsinghx/shl-2026-whisper-large-v3-transcript-data',
            'python': platform.python_version(), 'sklearn': sklearn.__version__,
            'pandas': pd.__version__, 'numpy': np.__version__}
(OUTPUT_DIR / 'experiment.json').write_text(json.dumps(manifest, indent=2))
print(f'Submission ready: {submission_path} | {len(submission)} rows')
display(submission.head())
display(FileLink(str(OUTPUT_ROOT / 'submission.csv')))

## 10. Experiment workspace — your next edits

Start with one change at a time and compare pooled six-fold RMSE:

1. Try Ridge alpha values `1`, `3`, `10`, `30`, `100` in the hyperparameter cell.
2. Compare word-only, character-only, and their combination.
3. Try lowercase on/off and character ranges `(2, 5)` or `(3, 6)`.
4. Add numeric features such as word count only inside the model pipeline.
5. Later replace this model with a pretrained text regressor, keeping the same folds.

Give each run a new `experiment_name` so its model, diagnostics, and predictions
are saved separately. Keep the transcript dataset unchanged. The score printed
here is local validation RMSE; the leaderboard score is a separate measurement.

After running on Kaggle, save the version with outputs and submit
`/kaggle/working/submission.csv` through the competition page. This notebook does
not automatically consume a submission attempt when you rerun experiments.

In [ ]:
# Add your next experiment or error-analysis code here.
# Available objects: CONFIG, train, test, cv_metrics, oof, final_model, submission.

## Paired Small vs Large-v3 results — completed local run

Recorded comparison with the original library versions, identical six folds and unchanged scoring hyperparameters. Large-v3 did not demonstrate an improvement: **Small RMSE 1.11795; Large-v3 RMSE 1.12624**.

| Fold | Small RMSE | Large-v3 RMSE | Difference |
|---|---:|---:|---:|
| 1 | 1.29625 | 1.31099 | +0.01475 |
| 2 | 1.16112 | 1.17862 | +0.01750 |
| 3 | 0.84703 | 0.84965 | +0.00262 |
| 4 | 1.16022 | 1.17306 | +0.01284 |
| 5 | 1.13288 | 1.13745 | +0.00457 |
| 6 | 1.05836 | 1.05232 | -0.00604 |

Large-v3 minus Small: **+0.00829**, approximate paired-record bootstrap 95% interval **[-0.00188, 0.01872]** from 20,000 resamples. The interval includes zero. Uncertainty is conditional on fixed OOF predictions; shared speakers and overlapping CV training sets are limitations.

Both transcript sets cover all 985 recordings. Normalized words changed in 909 recordings (92.3%); corpus word disagreement was 20.09%, using Small as a reference. This is **not word error rate** because neither transcript set is human ground truth. Repetition flags decreased from 18 to 14, but eight new flags appeared. Quality changes are mixed.

Post hoc error analysis: the 37 zero-rated rows worsened (3.19635 to 3.31438); the other 732 improved slightly (0.89251 to 0.88163). Keep the overall result as the primary comparison. No rows were dropped.

The results above record the completed local run. Editing or rerunning this notebook does not update this historical comparison automatically.


### Submitted public leaderboard result

The full-training-set model generated 216 test predictions. The submission was accepted on 9 October 2026: **Large-v3 public score 0.6522; Small public score 0.6607** (lower is better). Public score improved by 0.0085, while local OOF RMSE worsened slightly. The evaluation sets differ; the local bootstrap interval does not test the public-score difference. Evidence for a consistent downstream gain is mixed.


In [ ]:
# Recorded comparison from this completed run.
comparison = {'rows': 769, 'small_oof_rmse': 1.1179543360390638, 'large_v3_oof_rmse': 1.1262422774494176, 'delta_rmse_large_minus_small': 0.008287941410353827, 'paired_record_bootstrap_95_percent_interval': [-0.0018791507583629018, 0.018718821862338562], 'bootstrap_repetitions': 20000, 'seed': 42, 'interpretation': 'Negative delta favors Large-v3; positive delta favors Small.', 'limitation': 'Approximate uncertainty conditional on these fixed OOF predictions. Rows may share speakers, and cross-validation training sets overlap. This is not proof of a universal model-size effect or a test of ASR word accuracy.', 'new_transcripts_sha256': '24a05df69ed5a5351edef4bc6fdf980f681dd89c52c671dd03363a1db82069f3', 'changed_training_transcripts': 754}
display(pd.Series(comparison).to_frame("value"))